# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

For **Refresh / Content Opportunity Scoring**, one row in this dataset represents the aggregated performance metrics for a **unique piece of content** (e.g., an article, a product page) over a **single day**. The appropriate time window for analysis would be the **last 90 days** to capture recent performance trends relevant for refresh decisions.

In [1]:
import pandas as pd
import numpy as np

# Define a date range for the sample data
dates = pd.date_range(start='2023-01-01', periods=100, freq='D')

# Create content IDs
content_ids = [f'content_{i}' for i in range(20)]

# Create a sample DataFrame
np.random.seed(42) # for reproducibility

data = []
for _ in range(200): # 200 rows of daily content performance
    content_id = np.random.choice(content_ids)
    date = np.random.choice(dates)
    impressions = np.random.randint(100, 10000)
    clicks = np.random.randint(5, impressions // 5)
    ctr = clicks / impressions if impressions > 0 else 0
    avg_position = np.random.uniform(1.0, 30.0)
    bounce_rate = np.random.uniform(0.1, 0.9)
    time_on_page = np.random.uniform(30, 300)
    word_count = np.random.randint(300, 2000)
    content_type = np.random.choice(['article', 'product_page', 'blog_post'])
    last_modified_date = date - pd.to_timedelta(np.random.randint(0, 365), unit='D')
    creation_date = last_modified_date - pd.to_timedelta(np.random.randint(0, 730), unit='D')
    content_title = f"Title for {content_id}"
    url = f"https://example.com/{content_id}"
    author = np.random.choice(['Author A', 'Author B', 'Author C'])

    # Simulate some missing values
    if np.random.rand() < 0.05: # 5% chance of missing impressions
        impressions = np.nan
    if np.random.rand() < 0.03: # 3% chance of missing clicks
        clicks = np.nan
    if np.random.rand() < 0.01: # 1% chance of missing bounce_rate
        bounce_rate = np.nan

    # Refresh opportunity score (hypothetical)
    # Let's say content with low CTR and high impressions and older last modified date has higher refresh opportunity
    # Handle potential division by zero for ctr
    effective_ctr = ctr if ctr > 0 else 0.001
    refresh_opportunity_score = (1 / (effective_ctr + 0.01)) * (impressions / 1000) + (date - last_modified_date).days / 100
    refresh_opportunity_score = max(0, min(100, refresh_opportunity_score))

    data.append([
        content_id, date, impressions, clicks, ctr, avg_position, bounce_rate,
        time_on_page, word_count, content_type,
        last_modified_date, creation_date, content_title, url, author,
        refresh_opportunity_score
    ])

df_sample = pd.DataFrame(data, columns=[
    'content_id', 'date', 'impressions', 'clicks', 'ctr', 'average_position',
    'bounce_rate', 'time_on_page', 'word_count', 'content_type',
    'last_modified_date', 'creation_date', 'content_title', 'url', 'author',
    'refresh_opportunity_score'
])

# Ensure date columns are datetime objects
df_sample['date'] = pd.to_datetime(df_sample['date'])
df_sample['last_modified_date'] = pd.to_datetime(df_sample['last_modified_date'])
df_sample['creation_date'] = pd.to_datetime(df_sample['creation_date'])

# Sort by content_id and date for better representation
df_sample = df_sample.sort_values(by=['content_id', 'date']).reset_index(drop=True)

print("Sample DataFrame created successfully.")


Sample DataFrame created successfully.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Here's a classification of hypothetical fields relevant to content opportunity scoring:

### Features
These are the input variables used to predict the label.
-   `last_modified_date`: Date the content was last updated (recency, freshness).
-   `impressions`: Number of times the content was shown in search results (visibility).
-   `clicks`: Number of times users clicked on the content (engagement).
-   `ctr`: Click-through rate (clicks/impressions, effectiveness).
-   `average_position`: Average position in search results (ranking).
-   `bounce_rate`: Percentage of users who left the page after viewing only one page (user experience).
-   `time_on_page`: Average time users spent on the page (engagement).
-   `word_count`: Length of the content (proxy for depth/completeness).
-   `content_type`: Type of content (e.g., 'article', 'product_page', 'blog_post') (categorical attribute).

### Label
This is the output variable we are trying to predict or optimize.
-   `refresh_opportunity_score`: A continuous score indicating the potential benefit or urgency of refreshing the content. Higher scores suggest greater opportunity.

### Context
These fields provide additional information but are not directly used as features for the model; they help in understanding the content or debugging.
-   `content_id`: Unique identifier for the content.
-   `content_title`: Title of the content.
-   `url`: URL of the content.
-   `author`: Author of the content.
-   `creation_date`: Original creation date of the content.
-   `date`: The specific day for which the performance metrics are aggregated.

### Excluded
These fields are not included in the dataset for various reasons.
-   `internal_tracking_id`: An internal system ID that offers no predictive value for external content performance.
-   `user_ip_address`: Personally identifiable information (PII) that is not relevant for content scoring and should be excluded for privacy reasons.
-   `raw_search_query`: While relevant for detailed analysis, it's often too granular and high-cardinality to be directly used as a feature for broad content scoring; insights from it would be aggregated into other metrics if used.


In [2]:
# 3.1. Dataset Row Count
print(f"Total number of rows in the dataset: {len(df_sample)}")

# 3.2. Missing Values
print("\nMissing values per column:")
display(df_sample.isnull().sum().to_frame(name='missing_count'))


Total number of rows in the dataset: 200

Missing values per column:


,missing_count
content_id,0
date,0
impressions,10
clicks,5
ctr,0
average_position,0
bounce_rate,2
time_on_page,0
word_count,0
content_type,0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
# 3.3. Date Range
min_date = df_sample['date'].min()
max_date = df_sample['date'].max()
print(f"\nDate range of the dataset: {min_date.strftime('%Y-%m-%d')} to {max_date.strftime('%Y-%m-%d')}")

# 3.4. Preview of the Dataset
print("\nFirst 5 rows of the dataset:")
display(df_sample.head())



Date range of the dataset: 2023-01-01 to 2023-04-10

First 5 rows of the dataset:


,content_id,date,impressions,clicks,ctr,average_position,bounce_rate,time_on_page,word_count,content_type,last_modified_date,creation_date,content_title,url,author,refresh_opportunity_score
0,content_0,2023-01-08,7568.0,347.0,0.045851,13.482395,0.353557,147.340909,858,blog_post,2022-01-13,2020-03-26,Title for content_0,https://example.com/content_0,Author B,100.000000
1,content_0,2023-01-13,6456.0,246.0,0.038104,19.249665,0.259085,47.446504,971,article,2022-07-10,2021-06-11,Title for content_0,https://example.com/content_0,Author C,100.000000
2,content_0,2023-01-18,8515.0,1267.0,0.148796,5.362090,0.575305,132.840531,1126,blog_post,2022-07-26,2021-04-22,Title for content_0,https://example.com/content_0,Author A,55.382176
3,content_0,2023-02-20,763.0,63.0,0.082569,10.372885,0.515032,219.815119,1436,product_page,2022-12-31,2022-04-08,Title for content_0,https://example.com/content_0,Author B,8.752517
4,content_0,2023-02-20,4880.0,721.0,0.147746,7.009602,0.802577,234.389766,333,blog_post,2022-12-11,2021-06-09,Title for content_0,https://example.com/content_0,Author B,31.645827


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Here are some realistic limitations of this data for **Refresh / Content Opportunity Scoring**:

1.  **Limited History**: The dataset might only cover a short period (e.g., 90 days), making it difficult to identify long-term trends, seasonal patterns, or the historical impact of past content refreshes. This can lead to short-sighted refresh recommendations.
2.  **Attribution Challenges**: It's hard to isolate the precise impact of a content refresh. Changes in performance might be due to external factors like algorithm updates, competitor actions, broader market trends, or other marketing efforts, rather than solely the refresh itself.
3.  **Bias in Historical Data**: Past performance metrics may not perfectly predict future opportunity. Content that performed well historically might have diminishing returns if refreshed, while new topics or formats that lack historical data could be overlooked, leading to an 'exploit vs. explore' dilemma.
4.  **Lack of Qualitative Data**: The dataset primarily consists of quantitative engagement metrics. It doesn't capture qualitative aspects like content quality, factual accuracy, user sentiment from comments/reviews, or design effectiveness, which are crucial for true content opportunity assessment.
5.  **Granularity Limitations**: If data is aggregated at a daily content level, it might mask finer-grained nuances (e.g., hourly performance spikes, performance differences across specific user segments or devices) that could be relevant for highly optimized refresh strategies.

In [4]:
# No code required for this section based on the prompt's request for markdown explanation.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.